# ANN-Based Customer Churn Prediction
Instructor Answer Key Notebook

## Import Required Libraries

In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout


ModuleNotFoundError: No module named 'tensorflow'

## Load Dataset

In [ ]:

df = pd.read_csv("customer_churn.csv")
df.head()


In [ ]:
df.shape

In [ ]:
df.info()

## Explore Target Variable

In [ ]:

df["Churn"].value_counts()


In [ ]:

sns.countplot(x="Churn", data=df)
plt.title("Customer Churn Distribution")
plt.show()


## Drop Unnecessary Columns

In [ ]:

df = df.drop(["customerID"], axis=1)
df.head()


## Handle Missing Values

In [ ]:
df.isnull().sum()

In [ ]:
df = df.dropna()

## Separate Features and Target

In [ ]:

X = df.drop("Churn", axis=1)
y = df["Churn"]


## Encode Target Variable

In [ ]:

y = y.map({"No":0, "Yes":1})
y.head()


## Identify Numerical and Categorical Columns

In [ ]:

cat_cols = X.select_dtypes(include=["object"]).columns
num_cols = X.select_dtypes(exclude=["object"]).columns

print("Categorical columns:", cat_cols)
print("Numerical columns:", num_cols)


## Preprocessing Pipeline

In [ ]:

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])


## Train-Test Split

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


## Apply Preprocessing

In [ ]:

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)


In [ ]:
X_train_processed.shape, X_test_processed.shape

## Build ANN Model

In [ ]:

model = Sequential([
    Dense(16, activation="relu", input_shape=(X_train_processed.shape[1],)),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])


## Compile Model

In [ ]:

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


## Train Model

In [ ]:

history = model.fit(
    X_train_processed,
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)


## Plot Training Accuracy

In [ ]:

plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Model Accuracy")
plt.legend()
plt.show()


## Plot Training Loss

In [ ]:

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Model Loss")
plt.legend()
plt.show()


## Predictions

In [ ]:

y_pred_prob = model.predict(X_test_processed)
y_pred = (y_pred_prob > 0.5).astype(int)


## Evaluate Model

In [ ]:

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))


In [ ]:

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Purples")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:

print(classification_report(y_test, y_pred))
